# Independent Clean CV-DV LCHS Demo

This notebook imports only the new `clean_*` modules. It treats the working paper as a hypothesis rather than ground truth and checks the implementation against exact linear algebra wherever possible.

The main idea is to separate three different sources of error that can otherwise get mixed together:
- kernel/truncation error: the target CV construction may already differ from the exact PDE solution
- state-preparation error: `snap_d` may fail to prepare the desired CV coefficient state exactly
- circuit/Trotter/readout error: the hybrid evolution and postselection may differ from the ideal truncated CV model

Sections:
- Dirichlet heat-equation benchmark
- convention and Pauli reconstruction checks
- coefficient backend agreement
- exact-vs-Trotter validation
- state-prep/readout comparison across `injection`, `snap_d`, and `givens`
- generic Pauli-system template
- practical workflow for tuning PDE fidelity and CV state-prep fidelity


## How To Read The Metrics

The runtime reports several fidelities because they answer different questions:

- `fidelity_vs_exact`: compares the observed DV output to the exact discretized PDE solution vector `exp(-A T) u_0`
- `fidelity_vs_truncated`: compares the observed DV output to the ideal finite-Fock CV model with the same kernel settings
- `oracle_fidelity`: compares the prepared CV state to the target LCHS coefficient state before the hybrid evolution
- `postselection_probability`: probability that the final CV postselection onto the squeezed-vacuum-derived `|0>` slice succeeds

Interpretation:
- If `fidelity_vs_truncated` is close to `1` but `fidelity_vs_exact` is not, then the main gap is in the kernel/truncation model, not in the circuit execution.
- If `oracle_fidelity` is low, then the CV state-preparation method is the bottleneck.
- If `oracle_fidelity` is high but `fidelity_vs_truncated` is low, then Trotterization, compilation, or readout is the bottleneck.

Method caveats used in this notebook:
- `injection` is the ideal simulator baseline and isolates the PDE-side approximation error.
- `snap_d` is the actual gate-level CV preparation ansatz currently implemented in the backend circuit.
- `givens` is exact in simulation at the statevector level, but its current runtime realization is still direct injection plus a hardware-resource report, not an explicit JC-pulse circuit.


In [ ]:
import numpy as np

from clean_core import (
    EvolutionSpec,
    KernelSpec,
    StatePrepSpec,
    build_dirichlet_heat_system,
    build_pauli_system,
    coefficient_backend_gap,
    exact_reference_map,
    generator_matrix,
)
from clean_hybrid import run_clean_lchs


## 1. Dirichlet Heat Equation Setup

We use the semidiscrete 1D homogeneous Dirichlet heat equation

$$
\frac{d u}{d t} = -A u,
$$

with

$$
A = \frac{\alpha}{h^2} \operatorname{tridiag}(-1, 2, -1).
$$

For `num_qubits = 2`, the DV register has dimension `2^2 = 4`, so the benchmark acts on a four-component interior-grid state vector. The clean stack decomposes this dense matrix into Pauli terms and then rebuilds the hybrid evolution from that Pauli representation.


In [ ]:
# PDE benchmark hyperparameters.
# num_qubits sets the grid dimension through 2**num_qubits.
num_qubits = 2
alpha = 1.0
grid_spacing = 1.0
total_time = 1.0
init_basis_index = 1

system = build_dirichlet_heat_system(
    num_qubits=num_qubits,
    alpha=alpha,
    grid_spacing=grid_spacing,
    total_time=total_time,
    init_basis_index=init_basis_index,
)

A = generator_matrix(system)
reference_map = exact_reference_map(system)
reference_vector = reference_map @ system.init_state

print('System label:', system.label)
print('Grid dimension:', system.dv_dim)
print('Initial DV state u0:', system.init_state)
print()
print('Generator matrix A = L + iH:')
print(A)
print()
print('Exact reference map exp(-AT):')
print(reference_map)
print()
print('Exact discretized heat-equation solution vector exp(-AT) u0:')
print(reference_vector)
print()
print('Pauli decomposition (L block):')
for term in system.l_terms:
    print(f'  {term.label}: {term.coeff.real:+.8f}{term.coeff.imag:+.8f}j')


### What The Benchmark Cell Prints

The benchmark cell prints three things:
- the dense benchmark generator `A = L + iH`
- the exact DV reference map `exp(-A T)`
- the exact discretized solution vector `exp(-A T) u_0`

For the heat benchmark, `H = 0`, so `A = L` is real and symmetric. That makes this a useful sanity check that the Pauli reconstruction matches the expected tridiagonal heat operator.


## 2. Convention and Coefficient Checks

The coefficient-generation step is where many sign and convention mistakes usually hide. This section therefore checks the numerical kernel before we interpret any PDE fidelity numbers.

What is being tested here:
- `coefficient_backend_gap(kernel)`: agreement between the direct overlap quadrature and the compensated Gauss-Hermite backend
- internal operator convention: the clean stack uses `hbar = 1` and `x_hat = (a + a^\dagger) / \sqrt{2}`

A small backend gap means the coefficient vector is numerically stable for the chosen hyperparameters. If this gap becomes large, increase `n_quad`, lower the aggressiveness of the squeezing window, or both.


In [ ]:
kernel = KernelSpec(
    r_target=1.1,
    r_prime=0.1,
    beta=0.75,
    n_coeff=12,
    n_fock=32,
    n_quad=180,
    coeff_backend='explicit_overlap',
)

print('KernelSpec:', kernel)
print('Coefficient backend gap (explicit vs Gauss-Hermite):', coefficient_backend_gap(kernel))
print('Internal convention: hbar=1, x_hat=(a+a^dagger)/sqrt(2)')
print('Interpretation: a small backend gap means the coefficient vector is numerically stable for this kernel.')


## 3. State Preparation And Readout Comparison

This section separates state-preparation quality from PDE solution quality.

The three preparation paths are:
- `injection`: exact simulator seed loading, useful as the clean reference baseline
- `snap_d`: alternating SNAP plus displacement layers optimized in truncated Fock space
- `givens`: exact adjacent-level decomposition with direct state injection in simulation and hardware counts reported separately

What to watch for:
- compare `oracle_fidelity` across methods to judge CV preparation quality
- compare `fidelity_vs_exact` to judge final PDE solution quality
- compare `fidelity_vs_truncated` to see whether the backend circuit is faithfully realizing the same truncated CV model


### Why The State-Prep Table Matters

For tuning purposes, this table should usually be read in the following order:

1. `oracle_fidelity`: Did the CV method prepare the intended coefficient state?
2. `fidelity_vs_truncated`: Given that prepared state, did the hybrid circuit realize the same truncated CV model?
3. `fidelity_vs_exact`: After both of those steps, how close is the final PDE solution to the exact discretized target?

This ordering helps avoid the common mistake of blaming the circuit when the kernel or the state preparation is actually the dominant source of error.


In [ ]:
methods = ['injection', 'snap_d', 'givens']
evolution = EvolutionSpec(n_trotter_steps=20, readout_mode='postselect_statevector')

results = {}
for method in methods:
    prep = StatePrepSpec(method=method, snap_depth=3, snap_restarts=2, snap_maxiter=300)
    results[method] = run_clean_lchs(system, kernel, prep, evolution)

for method, result in results.items():
    print(f'[{method}]')
    print(f'  fidelity_vs_exact       = {result.fidelity_vs_exact:.8f}')
    print(f'  fidelity_vs_truncated   = {result.fidelity_vs_truncated:.8f}')
    print(f'  oracle_fidelity         = {result.oracle_fidelity:.8f}')
    print(f'  postselection_prob      = {result.postselection_probability:.6e}')
    print(f'  rel_error_vs_exact      = {result.rel_error_vs_exact:.6e}')
    print(f'  rel_error_vs_truncated  = {result.rel_error_vs_truncated:.6e}')
    print(f'  circuit_depth           = {result.circuit_depth}')
    print(f'  oracle_apply_mode       = {result.metadata.get("oracle_apply_mode")}')
    print(f'  oracle_metadata         = {result.metadata.get("oracle_metadata")}')
    print()


## 4. Exact-Vs-Trotter Comparison

This section uses `injection` on purpose so that CV state-preparation error is removed from the comparison. The goal is to understand whether increasing the Trotter step count actually improves the implemented hybrid evolution.

Interpretation:
- if `fidelity_vs_truncated` improves strongly with more Trotter steps, then circuit discretization error is real
- if `fidelity_vs_exact` barely changes while `fidelity_vs_truncated` improves, then the remaining gap is mostly kernel/truncation error


In [ ]:
trotter_grid = [5, 10, 20, 40]
prep = StatePrepSpec(method='injection')
for steps in trotter_grid:
    evo = EvolutionSpec(n_trotter_steps=steps, readout_mode='postselect_statevector')
    res = run_clean_lchs(system, kernel, prep, evo)
    print(f'n_trotter_steps={steps:3d}')
    print(f'  fidelity_vs_exact       = {res.fidelity_vs_exact:.8f}')
    print(f'  fidelity_vs_truncated   = {res.fidelity_vs_truncated:.8f}')
    print(f'  rel_error_vs_exact      = {res.rel_error_vs_exact:.6e}')
    print(f'  rel_error_vs_truncated  = {res.rel_error_vs_truncated:.6e}')
    print()


## 5. Generic Pauli-System Template

This section is the minimal entry point for trying a non-heat benchmark.

You provide:
- `l_terms`: Hermitian Pauli terms for `L`
- `h_terms`: Hermitian Pauli terms for `H`
- `total_time`: evolution time `T`
- `init_state`: arbitrary DV initial statevector

The runtime then constructs the effective generator

$$
A = L + i H
$$

and compares the clean hybrid implementation to `exp(-A T)`.


In [ ]:
# In the generic template below, l_terms and h_terms should each be Hermitian
# Pauli sums. The runtime will interpret them as A = L + iH.
generic_system = build_pauli_system(
    l_terms=[('I', 0.15 + 0.0j)],
    h_terms=[('Y', -0.8 + 0.0j)],
    total_time=0.7,
    init_state=np.array([1.0, 1.0j]),
    label='generic_single_qubit_demo',
)
generic_kernel = KernelSpec(
    r_target=1.0,
    r_prime=0.15,
    beta=0.7,
    n_coeff=10,
    n_fock=32,
    n_quad=160,
)
generic_prep = StatePrepSpec(method='injection')
generic_evo = EvolutionSpec(n_trotter_steps=20, readout_mode='postselect_statevector')
generic_result = run_clean_lchs(generic_system, generic_kernel, generic_prep, generic_evo)
print('Generic system fidelity_vs_exact:', generic_result.fidelity_vs_exact)
print('Generic system fidelity_vs_truncated:', generic_result.fidelity_vs_truncated)
print('Generic system postselection_probability:', generic_result.postselection_probability)
print('Generic system rel_error_vs_exact:', generic_result.rel_error_vs_exact)


## 6. Practical Workflow For Tuning

A reliable tuning workflow is:

1. Start with `injection` and optimize for `fidelity_vs_exact` to identify a strong kernel region.
2. Freeze those kernel parameters and compare `snap_d` against `injection` and `givens`.
3. Increase `snap_depth`, `snap_restarts`, and `snap_maxiter` only if `oracle_fidelity` is the limiting factor.
4. Increase `n_trotter_steps` only if `fidelity_vs_truncated` is clearly below `1`.

In short:
- poor `oracle_fidelity` means state-prep work is needed
- poor `fidelity_vs_truncated` means hybrid compilation or Trotter work is needed
- poor `fidelity_vs_exact` with high truncated fidelity means the kernel itself needs retuning
